# 🚀 Artemis VLM Pipeline: Complete Walkthrough

This notebook demonstrates the full pipeline:
1. **Load sample from SQL** (vlm_sample table)
2. **Router prediction** (which VLM is best?)
3. **Load Balancer scheduling** (SLA/capacity aware)
4. **Inference** (actual VLM call)

```
SQL Sample → Router → Load Balancer → Inference → Response
```

## 1️⃣ Setup: Path Configuration

In [16]:
import sys
from pathlib import Path
import time

# Add project root to path for package imports
PROJECT_ROOT = Path.cwd().parent.parent.parent  # Which_VLM_Router/  # Which_VLM_Router/
sys.path.insert(0, str(PROJECT_ROOT))

CHECKPOINTS_DIR = PROJECT_ROOT / 'artemis_final' / 'checkpoints'

print(f'Project root: {PROJECT_ROOT}')
print(f'Checkpoints: {CHECKPOINTS_DIR}')
print('✓ Paths configured')

Project root: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router
Checkpoints: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final/checkpoints
✓ Paths configured


## 2️⃣ Load Sample from SQL Database

In [17]:
from sqlalchemy import create_engine, text
import base64
from io import BytesIO

# Import DB config from ares
from artemis_final.ares.configs.db_config import DB_URL, TABLES

def load_random_sample_with_image():
    """Load a random sample WITH image from database."""
    try:
        engine = create_engine(DB_URL)
        
        with engine.connect() as conn:
            # Join vlm_samples with vlm_images to get sample + image together
            result = conn.execute(text(f"""
                SELECT 
                    s.sample_id, 
                    s.prompt_text,
                    s.router_task, 
                    s.source_dataset,
                    s.ground_truth,
                    i.image_bytes,
                    i.img_width,
                    i.img_height
                FROM {TABLES['samples']} s
                LEFT JOIN {TABLES['images']} i ON s.image_id = i.image_id
                WHERE s.prompt_text IS NOT NULL
                  AND i.image_bytes IS NOT NULL
                ORDER BY RANDOM()
                LIMIT 1
            """))
            row = result.fetchone()
        
        if row:
            return {
                'sample_id': row[0],
                'prompt': row[1],
                'router_task': row[2] or 'vqa',
                'source_dataset': row[3] or 'unknown',
                'ground_truth': row[4],
                'image_bytes': row[5],  # Binary image data
                'img_width': row[6],
                'img_height': row[7],
            }
        return None
    except Exception as e:
        print(f'⚠️ Database error: {e}')
        return None

# Try loading from SQL
sample = load_random_sample_with_image()

if sample is None:
    sample = {
        'sample_id': 'synthetic_001',
        'prompt': 'Extract all text from this receipt image.',
        'router_task': 'ocr',
        'source_dataset': 'synthetic',
        'ground_truth': None,
        'image_bytes': None,
        'img_width': None,
        'img_height': None,
    }
    print('⚠️ Using synthetic sample (no DB connection or no samples with images)')
else:
    print('✓ Loaded sample WITH image from SQL')

print(f'\n📋 Sample:')
print(f'   ID: {sample["sample_id"]}')
print(f'   Task: {sample["router_task"]}')
print(f'   Dataset: {sample["source_dataset"]}')
print(f'   Prompt: {sample["prompt"][:80]}...')
if sample['image_bytes']:
    print(f'   Image: {sample["img_width"]}x{sample["img_height"]} ({len(sample["image_bytes"])/1024:.1f} KB)')
else:
    print('   Image: None')

✓ Loaded sample WITH image from SQL

📋 Sample:
   ID: robut_wikisql_623_1042fa1c
   Task: table_reasoning
   Dataset: cauldron
   Prompt: What is the extortion and theft rates where the United Nations Observer Mission ...
   Image: 699x520 (55.4 KB)


## 3️⃣ Router: Predict Best Model

In [18]:
import torch
from artemis_final.router.artemis_router import ClassicalRouterInference
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f'🚀 Using CUDA: {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
    print('🍎 Using Apple MPS')
else:
    DEVICE = 'cpu'
    print('💻 Using CPU')
# Load router
router = ClassicalRouterInference(
    checkpoint_path=str(CHECKPOINTS_DIR / 'best_classical_router.pt'),
    device=DEVICE,
    verbose=True
)

print(f'\n📊 Available models: {router.model_names}')
print(f'📊 Available modes: {router.mode_names}')

🍎 Using Apple MPS
[INFO] Loading ClassicalRouter from: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final/checkpoints/best_classical_router.pt
[INFO] Model loaded on device: mps
[INFO] Models: ['deepseek_ocr', 'gemma_3_27b', 'qwen2_5_vl_3b', 'qwen2_5_vl_7b', 'qwen3_vl_8b_thinking']
[INFO] Modes: ['accuracy', 'balanced', 'cheap', 'fast']

📊 Available models: ['deepseek_ocr', 'gemma_3_27b', 'qwen2_5_vl_3b', 'qwen2_5_vl_7b', 'qwen3_vl_8b_thinking']
📊 Available modes: ['accuracy', 'balanced', 'cheap', 'fast']


In [19]:
# Route the sample
router_result = router.route(
    prompt=sample['prompt'],
    mode='balanced',  # or 'accuracy', 'cheap', 'fast'
    metadata={
        'router_task': sample['router_task'],
        'source_dataset': sample.get('source_dataset', 'unknown')
    }
)

# Get router confidence (max probability = confidence in chosen model)
max_prob = max(router_result['probs'].values())

print('🔮 Router Prediction:')
print(f'   Chosen model: {router_result["chosen_model"]}')
print(f'   Router confidence: {max_prob:.3f}')  # Max probability = confidence
print(f'   Inference time: {router_result["inference_ms"]:.1f}ms')
print(f'   Probabilities:')
for model, prob in sorted(router_result['probs'].items(), key=lambda x: -x[1]):
    marker = '→' if model == router_result['chosen_model'] else ' '
    print(f'     {marker} {model}: {prob:.3f}')

🔮 Router Prediction:
   Chosen model: gemma_3_27b
   Router confidence: 0.360
   Inference time: 22.8ms
   Probabilities:
     → gemma_3_27b: 0.360
       qwen2_5_vl_3b: 0.325
       qwen2_5_vl_7b: 0.202
       qwen3_vl_8b_thinking: 0.079
       deepseek_ocr: 0.033


## 4️⃣ Load Balancer: SLA-Aware Scheduling

In [20]:
from artemis_final.load_balancer import (
    ArtemisLoadBalancer,
    RouterOutput,
    SchedulingContext,
    StatsRegistry,
)
from artemis_final.load_balancer.config import load_capacity_config

# Load model capacity configs
model_configs = load_capacity_config()

print('📊 Loaded model configs:')
for name, cfg in model_configs.items():
    print(f'   {name}: latency={cfg.base_latency_ms}ms, SLA={cfg.sla_ms}ms')

📊 Loaded model configs:
   deepseek_ocr: latency=300ms, SLA=1500ms
   qwen2_5_vl_3b: latency=400ms, SLA=1500ms
   qwen2_5_vl_7b: latency=800ms, SLA=2500ms
   qwen3_vl_8b_thinking: latency=1200ms, SLA=3000ms
   gemma_3_27b: latency=2000ms, SLA=4000ms


In [21]:
# Create stats registry (synthetic for now - replace with real stats from ares)
synthetic_stats = {
    'ocr': {
        'deepseek_ocr': {'avg_latency_ms': 300, 'avg_accuracy': 0.95},
        'qwen2_5_vl_3b': {'avg_latency_ms': 400, 'avg_accuracy': 0.88},
        'qwen2_5_vl_7b': {'avg_latency_ms': 800, 'avg_accuracy': 0.92},
        'qwen3_vl_8b_thinking': {'avg_latency_ms': 1200, 'avg_accuracy': 0.90},
        'gemma_3_27b': {'avg_latency_ms': 2000, 'avg_accuracy': 0.85},
    },
    'vqa': {
        'deepseek_ocr': {'avg_latency_ms': 350, 'avg_accuracy': 0.70},
        'qwen2_5_vl_3b': {'avg_latency_ms': 450, 'avg_accuracy': 0.82},
        'qwen2_5_vl_7b': {'avg_latency_ms': 900, 'avg_accuracy': 0.88},
        'qwen3_vl_8b_thinking': {'avg_latency_ms': 1400, 'avg_accuracy': 0.91},
        'gemma_3_27b': {'avg_latency_ms': 2200, 'avg_accuracy': 0.93},
    },
    # Add more task types as needed
}

stats_registry = StatsRegistry(synthetic_stats)

# Create load balancer
lb = ArtemisLoadBalancer(
    model_configs=model_configs,
    stats_registry=stats_registry,
    global_latency_sla_ms=2000.0,
    max_accuracy_drop=0.05,
    scheduling_mode='capacity_aware'
)

print('✓ Load Balancer created')

✓ Load Balancer created


In [22]:
# Convert router output to load balancer format
lb_input = RouterOutput(
    sample_id=sample['sample_id'],
    task_type=sample['router_task'],
    router_probs=router_result['probs'],
    preferred_model=router_result['chosen_model']
)

context = SchedulingContext(
    arrival_ts_ms=time.time() * 1000,
    load_profile='medium',
    metadata={'router_latency_ms': router_result['inference_ms']}
)

# Schedule
decision = lb.schedule(lb_input, context)

print('📋 Load Balancer Decision:')
print(f'   Router preferred: {decision.preferred_model}')
print(f'   LB chose: {decision.chosen_model}')
print(f'   Same model? {decision.preferred_model == decision.chosen_model}')
print(f'   Predicted latency: {decision.total_latency_ms:.1f}ms')
print(f'   Queue delay: {decision.queue_delay_ms:.1f}ms')
print(f'   SLA violated: {decision.sla_violated}')

Missing accuracy stats for task=table_reasoning, model=gemma_3_27b. Using default values. Check Ares aggregates to ensure stats are available.


📋 Load Balancer Decision:
   Router preferred: gemma_3_27b
   LB chose: gemma_3_27b
   Same model? True
   Predicted latency: 1000.0ms
   Queue delay: 0.0ms
   SLA violated: False


## 5️⃣ Inference: Call the VLM

In [23]:
# Check if inference client is available
try:
    # Use inference_engine module (now populated)
    from artemis_final.inference_engine import WhichVLMClient
    INFERENCE_AVAILABLE = True
    print('✓ WhichVLMClient available (from inference_engine)')
except ImportError as e:
    try:
        # Fallback to ares
        from artemis_final.ares.inference_api_call.client import WhichVLMClient
        INFERENCE_AVAILABLE = True
        print('✓ WhichVLMClient available (from ares)')
    except ImportError as e2:
        INFERENCE_AVAILABLE = False
        print(f'⚠️ Inference client not available: {e2}')

✓ WhichVLMClient available (from inference_engine)


In [24]:
# Load inference client from config
if INFERENCE_AVAILABLE:
    models_yaml = PROJECT_ROOT / 'artemis_final' / 'ares' / 'configs' / 'models.yaml'
    
    if models_yaml.exists():
        try:
            client = WhichVLMClient.from_yaml(str(models_yaml))
            print('✓ Loaded inference client')
            print(f'   VLM models: {client.list_vlm_models()}')
        except Exception as e:
            print(f'⚠️ Could not load client: {e}')
            client = None
    else:
        print(f'⚠️ Config not found: {models_yaml}')
        client = None
else:
    client = None

✓ Loaded inference client
   VLM models: ['deepseek_ocr', 'qwen2_5_vl_3b', 'qwen2_5_vl_7b', 'qwen3_vl_8b_thinking', 'gemma_3_27b']


In [25]:
import math

def compute_confidence_from_logprobs(logprobs_data):
    """
    Compute confidence score from logprobs.
    Uses average token probability as confidence proxy.
    """
    if not logprobs_data:
        return None, "no_logprobs"
    
    content = logprobs_data.get("content", [])
    if not content:
        return None, "empty_content"
    
    # Extract logprobs for each token
    token_probs = []
    for token_info in content:
        if isinstance(token_info, dict):
            logprob = token_info.get("logprob")
            if logprob is not None:
                prob = math.exp(logprob)  # Convert log prob to probability
                token_probs.append(prob)
    
    if not token_probs:
        return None, "no_valid_tokens"
    
    # Confidence = geometric mean of token probabilities (or use first N tokens)
    # For simplicity, use average of first 10 tokens
    first_n = token_probs[:10]
    avg_prob = sum(first_n) / len(first_n)
    
    return avg_prob, "logprobs"

# Execute VLM inference with image
if client is not None:
    chosen_model = decision.chosen_model
    
    print(f'🚀 Calling VLM: {chosen_model}')
    print(f'   Prompt: {sample["prompt"][:60]}...')
    
    try:
        if sample['image_bytes']:
            # Convert bytes to PIL Image for the VLM client
            from PIL import Image
            image = Image.open(BytesIO(sample['image_bytes']))
            
            print(f'   Image: {image.width}x{image.height}')
            
            # Call VLM with image (request logprobs)
            result = client.vlm.run_image(
                image=image,
                text=sample['prompt'],
                models=[chosen_model],
                max_tokens=500,
                logprobs=True,
                top_logprobs=1
            )
            
            model_result = result.get(chosen_model, {})
            if model_result.get('ok'):
                print(f'\n✅ Response from {chosen_model}:')
                print(f'   Latency: {model_result["latency_ms"]:.0f}ms')
                
                # Compute confidence from logprobs
                logprobs_data = model_result.get('logprobs')
                confidence, source = compute_confidence_from_logprobs(logprobs_data)
                
                if confidence is not None:
                    print(f'   VLM Confidence: {confidence:.3f} (from {source})')
                else:
                    print(f'   VLM Confidence: N/A ({source})')
                
                # Token usage
                usage = model_result.get('usage', {})
                if usage:
                    print(f'   Tokens: {usage.get("prompt_tokens", "?")} in, {usage.get("completion_tokens", "?")} out')
                
                # Response text
                print(f'\n   📝 Response:')
                print(f'   {model_result["response_text"][:600]}')
                
                if sample.get('ground_truth'):
                    print(f'\n   📌 Ground Truth:')
                    print(f'   {sample["ground_truth"][:300]}')
            else:
                print(f'❌ Error: {model_result.get("error")}')
        else:
            print('⚠️ No image data - skipping VLM inference')
            print(f'   Would have called: {chosen_model}')
            
    except Exception as e:
        print(f'❌ Inference error: {e}')
        import traceback
        traceback.print_exc()
else:
    print('⚠️ Inference client not available')
    print(f'   Would have called: {decision.chosen_model} with image')

🚀 Calling VLM: gemma_3_27b
   Prompt: What is the extortion and theft rates where the United Natio...
   Image: 699x520

✅ Response from gemma_3_27b:
   Latency: 1458ms
   VLM Confidence: 0.955 (from logprobs)
   Tokens: 289 in, 52 out

   📝 Response:
   Based on the provided table, the extortion/theft rate where the United Nations Observer Mission Uganda-Rwanda is active is **unknown**. 

The table lists "unknown" for this category for the Rwanda Civil War, where this mission was deployed.

   📌 Ground Truth:
   Unknown.


## 📊 Pipeline Summary

| Step | Component | Output |
|------|-----------|--------|
| 1 | SQL | Sample with prompt + task |
| 2 | Router | Model probabilities + chosen model |
| 3 | Load Balancer | Final model (may differ) + SLA check |
| 4 | Inference | Actual VLM response |

In [26]:
# Print full pipeline summary
print('='*60)
print('PIPELINE SUMMARY')
print('='*60)
print(f'Sample ID: {sample["sample_id"]}')
print(f'Task Type: {sample["router_task"]}')
print(f'Prompt: {sample["prompt"][:60]}...')
print('-'*60)
print(f'Router chose: {router_result["chosen_model"]} (latency: {router_result["inference_ms"]:.1f}ms)')
print(f'Load Balancer chose: {decision.chosen_model}')
print(f'Predicted total latency: {decision.total_latency_ms:.1f}ms')
print(f'SLA violated: {decision.sla_violated}')
print('='*60)

PIPELINE SUMMARY
Sample ID: robut_wikisql_623_1042fa1c
Task Type: table_reasoning
Prompt: What is the extortion and theft rates where the United Natio...
------------------------------------------------------------
Router chose: gemma_3_27b (latency: 22.8ms)
Load Balancer chose: gemma_3_27b
Predicted total latency: 1000.0ms
SLA violated: False
